# DFC Neuron Steering

Steer specific DFC neurons and compare model outputs.

In [1]:
from helpers import (
    load_all, get_activations, generate_with_hook, generate_with_clamp,
    build_steering_vector, build_clamped_vector, apply_clamped_vector,
    top_features, partition_counts, feature_partition,
)
import torch
import torch.nn.functional as F

tokenizer, model_a, model_b, dfc, cfg = load_all()
device = "cuda:0"
layer_idx = cfg.layer

`torch_dtype` is deprecated! Use `dtype` instead!


Loading Model A (ToolRL)...


Loading weights:   0%|          | 0/435 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading Model B (Base)...


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading DFC CrossCoder...
[DFC] dict=16384 k=90 | A-excl=819 B-excl=819 shared=14746
DFC layout: A-exclusive [0:819], B-exclusive [819:1638], Shared [1638:16384]


## 1. Inspect Features for a Prompt

In [2]:
prompt = """{'Name': 'Country Information API', 'Description': 'Retrieve information about countries, including their capitals, currencies, and flags.', 'Parameters': {}}"""

acts = get_activations(prompt, tokenizer, model_a, model_b, layer_idx, device)
features = dfc.encode(acts)  # (1, dict_size)
f = features[0]

counts = partition_counts(f, dfc)
print(f"Prompt: {prompt!r}")
print(f"\nActive features: {(f > 0).sum().item()} / {dfc.dict_size}")
print(f"  A-exclusive (ToolRL): {counts['a_exclusive']}")
print(f"  B-exclusive (Base):   {counts['b_exclusive']}")
print(f"  Shared:               {counts['shared']}")
print()
top_features(f, dfc)

Prompt: "{'Name': 'Country Information API', 'Description': 'Retrieve information about countries, including their capitals, currencies, and flags.', 'Parameters': {}}"

Active features: 90 / 16384
  A-exclusive (ToolRL): 1
  B-exclusive (Base):   2
  Shared:               87

Rank    Feat     Value  Partition
---------------------------------------------
   1    7994   12.0568  Shared
   2   14090   10.8186  Shared
   3    9407   10.2552  Shared
   4   10345    9.5019  Shared
   5    8657    8.9310  Shared
   6    6459    8.8252  Shared
   7   12479    8.3777  Shared
   8    5826    8.3667  Shared
   9    9888    8.2094  Shared
  10   14287    8.0833  Shared
  11    8965    7.8825  Shared
  12    9794    7.7332  Shared
  13   10315    7.7225  Shared
  14   12032    7.4335  Shared
  15   12589    7.2672  Shared


## 2. Build Steering Vector

In [3]:
top_a = torch.topk(f[:dfc.a_end], k=3)
top_b = torch.topk(f[dfc.a_end:dfc.b_end], k=3)

print("Top A-exclusive (ToolRL):")
for v, i in zip(top_a.values.tolist(), top_a.indices.tolist()):
    print(f"  Feature {i}: {v:.4f}")

print("\nTop B-exclusive (Base):")
for v, i in zip(top_b.values.tolist(), top_b.indices.tolist()):
    print(f"  Feature {dfc.a_end + i}: {v:.4f}")

Top A-exclusive (ToolRL):
  Feature 326: 5.7040
  Feature 1: 0.0000
  Feature 0: 0.0000

Top B-exclusive (Base):
  Feature 1386: 6.2776
  Feature 1249: 5.3441
  Feature 819: 0.0000


In [8]:
# === EDIT THESE to experiment ===
feature_scales = {
    #top_a.indices[0].item(): 15.0,   # amplify top ToolRL feature
    20: 50, #
    665: 50,
    326: 50
}

print("Steering for Model A:")
steer_a = build_steering_vector(feature_scales, dfc, model_idx=0)

print("\nSteering for Model B:")
steer_b = build_steering_vector(feature_scales, dfc, model_idx=1)

Steering for Model A:
  Feature 20 (A-exclusive (ToolRL)): scale=+50.0, ||dec||=0.6123
  Feature 665 (A-exclusive (ToolRL)): scale=+50.0, ||dec||=0.6353
  Feature 326 (A-exclusive (ToolRL)): scale=+50.0, ||dec||=0.6426

  Combined steering vector norm: 55.9356

Steering for Model B:
  Feature 20 (A-exclusive (ToolRL)): scale=+50.0, ||dec||=0.0000
  Feature 665 (A-exclusive (ToolRL)): scale=+50.0, ||dec||=0.0000
  Feature 326 (A-exclusive (ToolRL)): scale=+50.0, ||dec||=0.0000

  Combined steering vector norm: 0.0000


## 2b. Build Clamped Vector

Test `build_clamped_vector` and `apply_clamped_vector` — the multi-feature clamping analogue of `build_steering_vector`.

In [6]:
# from helpers import build_clamped_vector, apply_clamped_vector
# import copy

# # --- Unit tests (no GPU generation needed) ---

# # 1. Basic shapes and verbose output
# feature_clamps = {20: 10.0, 665: -5.0}
# dirs, targets = build_clamped_vector(feature_clamps, dfc, model_idx=0, verbose=True)
# assert dirs.shape == (2, dfc.activation_dim), f"Expected (2, {dfc.activation_dim}), got {dirs.shape}"
# assert targets.shape == (2,), f"Expected (2,), got {targets.shape}"
# assert targets.tolist() == [10.0, -5.0]
# print("\n[PASS] shapes and targets correct")

# # 2. apply_clamped_vector sets projections to target values
# h = torch.randn(1, dfc.activation_dim, device=device)
# h_clamped = apply_clamped_vector(h, dirs, targets)

# # Check that projections onto each decoder dir now equal the targets
# for i in range(dirs.shape[0]):
#     d = dirs[i]
#     proj = (h_clamped[0] @ d) / (d @ d).clamp(min=1e-10)
#     assert torch.allclose(proj, targets[i], atol=1e-4), \
#         f"Feature {i}: proj={proj.item():.6f}, target={targets[i].item():.6f}"
# print("[PASS] projections match targets after clamping")

# # 3. Idempotence — applying twice should give the same result
# h_clamped2 = apply_clamped_vector(h_clamped, dirs, targets)
# assert torch.allclose(h_clamped, h_clamped2, atol=1e-4), "Clamping should be idempotent"
# print("[PASS] clamping is idempotent")

# # 4. Batched hidden states — should work on (..., d) tensors
# h_batch = torch.randn(4, 8, dfc.activation_dim, device=device)
# h_batch_clamped = apply_clamped_vector(h_batch, dirs, targets)
# assert h_batch_clamped.shape == h_batch.shape
# for b in range(4):
#     for s in range(8):
#         for i in range(dirs.shape[0]):
#             d = dirs[i]
#             proj = (h_batch_clamped[b, s] @ d) / (d @ d).clamp(min=1e-10)
#             assert torch.allclose(proj, targets[i], atol=1e-4)
# print("[PASS] batched clamping correct")

# # 5. Single-feature clamping matches generate_with_clamp's logic
# feat_idx, clamp_val = 20, 7.5
# dirs_single, targets_single = build_clamped_vector({feat_idx: clamp_val}, dfc, model_idx=0, verbose=False)
# h_test = torch.randn(1, dfc.activation_dim, device=device)

# # Manual clamping (same math as generate_with_clamp)
# decoder_dir = dfc.W_dec[feat_idx, 0].to(device)
# dir_norm_sq = (decoder_dir @ decoder_dir).clamp(min=1e-10)
# proj_manual = (h_test @ decoder_dir) / dir_norm_sq
# h_manual = h_test + (clamp_val - proj_manual.unsqueeze(-1)) * decoder_dir

# h_fn = apply_clamped_vector(h_test, dirs_single, targets_single)
# assert torch.allclose(h_manual, h_fn, atol=1e-4), "Single-feature should match generate_with_clamp math"
# print("[PASS] single-feature matches generate_with_clamp logic")

# print("\nAll tests passed!")

## 3. Compare Steered vs Unsteered

In [9]:
test_prompt = """What are the capitals of France, Germany, Japan?"""
print("=" * 70)
print(f"PROMPT: {test_prompt}")
print("=" * 70)

gen = lambda m, d=None: generate_with_hook(m, tokenizer, test_prompt, layer_idx, steering_delta=d)

print("\n--- Model A (ToolRL) - NO steering ---")
print(gen(model_a))

print("\n--- Model A (ToolRL) - WITH steering ---")
print(gen(model_a, steer_a))

print("\n--- Model B (Base) - NO steering ---")
print(gen(model_b))

print("\n--- Model B (Base) - WITH steering ---")
print(gen(model_b, steer_b))

PROMPT: What are the capitals of France, Germany, Japan?

--- Model A (ToolRL) - NO steering ---
 The capital of France is Paris. The capital of Germany is Berlin. The capital of Japan is Tokyo.

--- Model A (ToolRL) - WITH steering ---


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


 I'm afraid you've asked an impossible question. The answer is not France, Germany, or Japan.

France's capital is Paris.
Germany's capital is Berlin.
Japan's capital is Tokyo.
I do not have information on this topic. Please provide a specific question so that I may answer it for you.

--- Model B (Base) - NO steering ---


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


 1. Paris, France: Paris is the capital and largest city of France. It is located in the north-central part of the country and is renowned for its rich history, stunning architecture, and cultural landmarks such as the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral.

2. Berlin, Germany: Berlin is the capital and largest city of Germany. It is located in

--- Model B (Base) - WITH steering ---
 The capitals of France, Germany, and Japan are Paris, Berlin, and Tokyo, respectively.


## 4. Scale Sweep

In [10]:
sweep_prompt = "List food and their benefits in json format"
scales = [0, -20, -40, -50, -100]

# print(f"Feature {sweep_feat} ({feature_partition(sweep_feat, dfc)})")
print(f"Prompt: {sweep_prompt!r}")
print("=" * 70)

for scale in scales:
    feature_scales = {20: scale, 665: scale, 326: scale}
    d = build_steering_vector(feature_scales, dfc, model_idx=0)
    #delta = None if scale == 0 else scale * dfc.W_dec[sweep_feat, 0]
    out = generate_with_hook(model_a, tokenizer, sweep_prompt, layer_idx, steering_delta=d)
    print(f"\n[scale={scale:>3}] {out}")

Prompt: 'List food and their benefits in json format'
  Feature 20 (A-exclusive (ToolRL)): scale=+0.0, ||dec||=0.6123
  Feature 665 (A-exclusive (ToolRL)): scale=+0.0, ||dec||=0.6353
  Feature 326 (A-exclusive (ToolRL)): scale=+0.0, ||dec||=0.6426

  Combined steering vector norm: 0.0000

[scale=  0] 
Here is a JSON list of various foods along with their primary health benefits:

```json
{
  "foods": [
    {
      "name": "Apples",
      "benefits": [
        "Rich in fiber, which aids digestion and helps maintain healthy cholesterol levels.",
        "High in antioxidants like quercetin and phloridzin, which may help protect against chronic diseases
  Feature 20 (A-exclusive (ToolRL)): scale=-20.0, ||dec||=0.6123
  Feature 665 (A-exclusive (ToolRL)): scale=-20.0, ||dec||=0.6353
  Feature 326 (A-exclusive (ToolRL)): scale=-20.0, ||dec||=0.6426

  Combined steering vector norm: 22.3743

[scale=-20] 
Here is the requested information about various foods and their benefits, presented in J

## 4b. Feature Clamping (Neuron 20 — JSON Generation)

Clamping forces a DFC feature's projection to a fixed value at every forward pass, rather than adding a constant delta. This tests whether neuron 20 (A-exclusive, ToolRL) is responsible for JSON formatting.

In [ ]:
clamp_feat = 326
clamp_values = [50, 100]
clamp_prompts = [
    "What are the capitals of Ukraine, France, and USA?",
]

print(f"Feature {clamp_feat} ({feature_partition(clamp_feat, dfc)})")
print("=" * 70)

for prompt in clamp_prompts:
    print(f"\nPROMPT: {prompt!r}")
    print("-" * 70)

    print(f"\n  [no clamp] (Model A)")
    print(f"  {generate_with_hook(model_a, tokenizer, prompt, layer_idx)[:300]}")

    for cv in clamp_values:
        out = generate_with_clamp(
            model_a, tokenizer, prompt, layer_idx,
            feature_idx=clamp_feat, clamp_value=cv, dfc=dfc, model_idx=0,
        )
        print(f"\n  [clamp={cv:>3}] (Model A)")
        print(f"  {out[:300]}")

    print()

Feature 326 (A-exclusive (ToolRL))

PROMPT: 'What are the capitals of Ukraine, France, and USA?'
----------------------------------------------------------------------

  [no clamp] (Model A)
   The capital of Ukraine is Kyiv (also spelled Kiev).
The capital of France is Paris.
The capital of the United States (USA) is Washington, D.C.


NameError: name 'generate_with_multi_clamp' is not defined

In [14]:
# Test clamping on Model B (Base) — can we induce JSON formatting?
clamp_prompt_b = "What are the capitals of France, Germany, Japan?"
print(f"Feature {clamp_feat} ({feature_partition(clamp_feat, dfc)})")
print(f"PROMPT: {clamp_prompt_b!r}")
print("=" * 70)

print("\n--- Model B (Base) - NO clamp ---")
print(generate_with_hook(model_b, tokenizer, clamp_prompt_b, layer_idx))

for cv in [5, 20, 50, 100]:
    print(f"\n--- Model B (Base) - clamp={cv} ---")
    print(generate_with_clamp(
        model_b, tokenizer, clamp_prompt_b, layer_idx,
        feature_idx=clamp_feat, clamp_value=cv, dfc=dfc, model_idx=1,
    ))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Feature 326 (A-exclusive (ToolRL))
PROMPT: 'What are the capitals of France, Germany, Japan?'

--- Model B (Base) - NO clamp ---


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


 The capital of France is Paris, the capital of Germany is Berlin, and the capital of Japan is Tokyo.

--- Model B (Base) - clamp=5 ---


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


 
A. Paris, Berlin, Tokyo
B. Paris, Berlin, London
C. Rome, Berlin, Tokyo
D. Rome, Berlin, London
Answer:

A

Which of the following statements about the characteristics of the U.S. economy is true?
A. The U.S. has become a typical industrial country
B. The U.S. is a typical agricultural country
C

--- Model B (Base) - clamp=20 ---


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


 The capital of France is Paris, the capital of Germany is Berlin, and the capital of Japan is Tokyo.

--- Model B (Base) - clamp=50 ---


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


 The capitals of France, Germany, and Japan are Paris, Berlin, and Tokyo, respectively.

--- Model B (Base) - clamp=100 ---
 The capital of France is Paris, the capital of Germany is Berlin, and the capital of Japan is Tokyo.


## 5. Feature Ablation

In [ ]:
abl_prompt = "Please list some healthy foods along with their benefits in a JSON structured format."

acts = get_activations(abl_prompt, tokenizer, model_a, model_b, layer_idx, device)
feats = dfc.encode(acts)
recon_full = dfc.decode(feats)

# Ablate each partition
for name, start, end in [
    ("A-exclusive", 0, dfc.a_end),
    ("B-exclusive", dfc.a_end, dfc.b_end),
    ("Shared", dfc.b_end, dfc.dict_size),
    ("Nothing", 0, 0),
]:
    ablated = feats.clone()
    ablated[:, start:end] = 0
    recon_abl = dfc.decode(ablated)
    mse = F.mse_loss(recon_abl[0], acts[0]).item()
    print(f"Without {name:12s}  MSE={mse:.6f}")

Without A-exclusive   MSE=1.329843
Without B-exclusive   MSE=1.331603
Without Shared        MSE=1.148484
Without Nothing       MSE=1.326300


## 6. Ablation-Based Steering

In [ ]:
# What A-exclusive features contribute to Model A's representation
feats_no_a = feats.clone()
feats_no_a[:, :dfc.a_end] = 0
recon_no_a = dfc.decode(feats_no_a)

a_contribution = recon_full[0, 0] - recon_no_a[0, 0]  # (d,)
print(f"A-exclusive contribution norm: {a_contribution.norm():.4f}")

steer_scale = 15.0
ablation_steer = steer_scale * F.normalize(a_contribution, dim=0)

abl_test = "How do I check the weather?"
print(f"\nPrompt: {abl_test!r}")

print("\n--- Base model, no steering ---")
print(generate_with_hook(model_b, tokenizer, abl_test, layer_idx))

print("\n--- Base model, steered with A-exclusive direction ---")
print(generate_with_hook(model_b, tokenizer, abl_test, layer_idx, steering_delta=ablation_steer))

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


A-exclusive contribution norm: 4.6419

Prompt: 'How do I check the weather?'

--- Base model, no steering ---


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


 - check the weather - look at the weather - check the weather report - look at the weather report
Answer: look at the weather report

--- Base model, steered with A-exclusive direction ---
 To check the weather, you can use various online resources such as websites like Weather.com, AccuWeather, or the official weather service of your country. You can also use mobile apps like Dark Sky or The Weather Channel. Additionally, you can check the weather on your phone's weather widget or by asking voice assistants like Google Assistant or Siri.


## 7. Free-Form

In [ ]:
my_prompt = "Explain photosynthesis formatted"
my_scales = {
    # feature_idx: scale,
    20: 80,
    665: 80,
}

if my_scales:
    sv_a = build_steering_vector(my_scales, dfc, model_idx=0)
    sv_b = build_steering_vector(my_scales, dfc, model_idx=1)

    for name, model, sv in [("ToolRL", model_a, sv_a), ("Base", model_b, sv_b)]:
        print(f"\n--- {name}, no steer ---")
        print(generate_with_hook(model, tokenizer, my_prompt, layer_idx))
        print(f"\n--- {name}, steered ---")
        print(generate_with_hook(model, tokenizer, my_prompt, layer_idx, steering_delta=sv))
else:
    print("Add feature indices and scales to my_scales above, then re-run!")

  Feature 20 (A-exclusive (ToolRL)): scale=+80.0, ||dec||=0.6123
  Feature 665 (A-exclusive (ToolRL)): scale=+80.0, ||dec||=0.6353

  Combined steering vector norm: 72.0240
  Feature 20 (A-exclusive (ToolRL)): scale=+80.0, ||dec||=0.0000
  Feature 665 (A-exclusive (ToolRL)): scale=+80.0, ||dec||=0.0000

  Combined steering vector norm: 0.0000

--- ToolRL, no steer ---
 for an elementary school child
Photosynthesis is like a magical recipe that plants use to make their own food using sunlight. Imagine the plant is a chef and sunlight is its ingredient. The chef uses sunlight, water (which comes from the soil) and air (carbon dioxide from the air). With these ingredients, the plant mixes them up in its leaves and voila! It makes its own food and

--- ToolRL, steered ---


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


 with a leaf. In the process of photosynthesis, a leaf is formed. It is converted into an embryo and then it is. Photosynthesis is a leaf. The leaf is a leaf, and the leaf is not. The leaf is not the leaf, and the leaf is not the leaf.

The leaf is not, and the leaf is not the leaf.

The leaf is not, and the

--- Base, no steer ---


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


 like a paragraph
Photosynthesis is a fundamental biological process that allows plants, algae, and some bacteria to convert light energy into chemical energy in the form of glucose and other organic compounds. This process occurs primarily in the chloroplasts of plant cells, where specialized pigments called chlorophyll absorb sunlight. The absorbed light energy is used to drive the synthesis of ATP (adenosine triphosphate

--- Base, steered ---
 in a way that a 5th grader can understand.
Photosynthesis is a process that plants and some other organisms use to make their own food. It happens in the leaves of plants, which are like green factories that make food. The main ingredients for photosynthesis are sunlight, water, and carbon dioxide.

Here's how it works:

1. **Sunlight**: Plants get energy from the


NameError: name 'clamp_prompts_b' is not defined